# Phase 1 — Noise Seed-Swap (the gate)
**Question:** does the initial *noise* (seed) or the *text* prompt decide how many objects SDXL generates?

We cross the SAME seeds against every (count, object) prompt, score with the detector, then decompose the realized count into noise vs text.

**Runtime:** GPU.

In [ ]:
import os
if not os.path.exists('src'):
    !git clone https://github.com/serinaqin/T2I-Count-Anomaly.git
    %cd T2I-Count-Anomaly
!pip install -q -r requirements.txt
!pip install -q pytest groundingdino-py

In [ ]:
import sys; sys.path.insert(0, '.')
import pandas as pd, os
from src.prompts import generate_grid
from src.pipeline import load_sdxl, generate
from src.detector import Detector
from src.scoring import count_from_detections
from src.config import load_config
from src.analysis import (text_responsiveness, count_variance_decomposition,
                          per_seed_summary, flag_degenerate)

In [ ]:
cfg = load_config('configs/phase1.yaml')
grid = generate_grid(cfg.counts, cfg.objects, cfg.seeds)
print(len(grid), 'images:', len(cfg.counts), 'counts x',
      len(cfg.objects), 'objects x', len(cfg.seeds), 'seeds')

In [ ]:
pipe = load_sdxl()
det = Detector()

In [ ]:
# Generate the full seed-swap grid and score each image with the detector.
rows = []
for i, p in enumerate(grid):
    img = generate(pipe, p.text, p.seed, cfg.num_inference_steps)
    dets = det.detect(img, [p.obj])
    n = count_from_detections(dets, p.obj, cfg.score_threshold)
    rows.append({'obj': p.obj, 'count': p.count, 'seed': p.seed,
                 'realized_count': n})
    if (i + 1) % 20 == 0:
        print(f'{i+1}/{len(grid)}')
df = pd.DataFrame(rows)
os.makedirs('results', exist_ok=True)
df.to_csv('results/phase1_counts.csv', index=False)
df.head()

In [ ]:
# Flag collage/degenerate blow-ups (correct counts, pathological images)
df = flag_degenerate(df, max_requested=max(cfg.counts))
print('degenerate images:', int(df['degenerate'].sum()), 'of', len(df))
df_clean = df[~df['degenerate']].copy()

## Verdict metrics
- **text slope** ~1 = text controls the count; ~0 = text ignored.
- **variance decomposition** (eta^2): how much of the realized count aligns with seed vs count vs object.
- **per-seed summary**: a seed with low std across prompts has a fixed 'preferred count' (noise-driven).

In [ ]:
print('text responsiveness:', text_responsiveness(df_clean))
print('variance decomposition:', count_variance_decomposition(df_clean))
per_seed_summary(df_clean)

In [ ]:
# Money plot A: per-seed count response
import matplotlib.pyplot as plt
piv = df_clean.groupby(['seed', 'count'])['realized_count'].mean().reset_index()
fig, ax = plt.subplots(figsize=(6, 5))
for s, g in piv.groupby('seed'):
    ax.plot(g['count'], g['realized_count'], marker='o', alpha=0.6,
            label=f'seed {s}')
lims = [min(cfg.counts), max(cfg.counts)]
ax.plot(lims, lims, 'k--', label='y=x (perfect text control)')
ax.set_xlabel('requested count'); ax.set_ylabel('mean realized count')
ax.set_title('Per-seed response (flat = noise-fixed, diagonal = text-controlled)')
ax.legend(fontsize=7); plt.tight_layout()
plt.savefig('results/phase1_perseed.png', dpi=90, bbox_inches='tight'); plt.show()

In [ ]:
# Money plot B: variance decomposition
dec = count_variance_decomposition(df_clean)
fig, ax = plt.subplots(figsize=(4, 4))
ax.bar(list(dec.keys()), list(dec.values()))
ax.set_ylabel('variance explained (eta^2)'); ax.set_ylim(0, 1)
ax.set_title('What drives the realized count?')
plt.tight_layout()
plt.savefig('results/phase1_variance.png', dpi=90, bbox_inches='tight'); plt.show()

## How to read this
- **Flat per-seed lines + seed eta^2 >> count eta^2 + slope ~ 0** -> the NOISE decides the count on SDXL. Phase 2 then localizes *where/when* the noise's count signal is read out in the U-Net.
- **Diagonal lines + count eta^2 >> seed eta^2 + slope ~ 1** -> the TEXT controls the count and the noise-prior story does NOT transfer to SDXL; Phase 2 pivots to the text/cross-attention (matching) pathway.
- **In between** -> both matter; the relative eta^2 tells us how to weight the Phase 2 probes.